[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/kitaev_wire/kitaev_wire.ipynb)

# Majorana Modes in a Kitaev Wire

This example computes how a moving electrostatic gate transports a Majorana mode along a Kitaev wire, a superconducting chain whose topological segment binds a zero-energy Majorana mode at each end. Together the two end modes hold one fermionic excitation whose energy stays pinned near zero while the ends remain far apart. Shifting one gate boundary slowly carries its bound mode along intact, whereas driving the gate too quickly sheds weight into bulk excitations.

Each site's two real Majorana components are modelled as vectors `x` and `y` in the two-dimensional geometric algebra, and the wire's Hamiltonian acts as a skew extensor map over fields on the sites. Chemical potential turns a site's two components in the plane `xy` via `-potential * xy.commutator(Majorana)`, while nearest-neighbour hopping and p-wave pairing form `forward_couplings - forward_couplings.adjoint()` across bonds, realizing the commutator action of one bivector in the full Majorana Clifford algebra. Because the generator is skew-adjoint, midpoint Cayley steps `(Majorana - half_step).solve(state + half_step(state))` preserve the state norm and excitation energies follow from `generator.adjoint()(generator)`.

In second-quantized Bogoliubov-de Gennes notation ([Kitaev 2001](https://arxiv.org/abs/cond-mat/0010440); [Alicea et al. 2011](https://arxiv.org/abs/1006.4395)), the chain Hamiltonian reads $H = \sum_j\left[-t\,(a_j^\dagger a_{j+1} + a_{j+1}^\dagger a_j) - \mu_j\,(a_j^\dagger a_j - \tfrac12) + \Delta\,a_j a_{j+1} + \Delta^*\,a_{j+1}^\dagger a_j^\dagger\right]$, which splits each complex fermion into two Majorana operators $c_{2j-1} = a_j + a_j^\dagger$ and $c_{2j} = (a_j - a_j^\dagger)/i$ with imaginary factors and alternating odd-even site indices, $H = \tfrac{i}{2}\sum_j\left[-\mu_j\,c_{2j-1}c_{2j} + (t + |\Delta|)\,c_{2j}c_{2j+1} + (|\Delta| - t)\,c_{2j-1}c_{2j+2}\right]$, or into a doubled $2N \times 2N$ particle-hole matrix that mixes creation and annihilation blocks.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import save_figure, show_animation, show_states
from examples.quantum.kitaev_wire import render

ga = VGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                     # Scalar+
# x and y label the two Majorana components at each site, not spatial directions.
Majorana = ga.gatype.vector()                                   # Vector+

## Bond couplings and the wire generator

On each site the chemical potential turns the two Majorana components in `xy`, while hopping and p-wave pairing couple components across each nearest-neighbour bond. Subtracting the adjoint of the forward bond couplings makes the full generator skew-adjoint.

In [ ]:
n_sites = 96
sites = np.arange(n_sites)                                      # [n_sites]
hopping = 1.0
pairing = 0.8
outside_potential = 3.5
inside_potential = 1.4
edge_width = 1.2
left_end = 18.0
right_end = 82.0

In [ ]:
def chain_couplings(n_sites: int, hopping: float, pairing: float) -> SparseExtensor:
    sites = np.arange(n_sites)                                  # [n_sites]
    # Hopping and p-wave pairing add in one cross-site coupling and subtract in the other.
    # A directed bond reads one Majorana component on the next site and returns one on this site.
    bond_coupling = ((hopping + pairing) * mv.y * (mv.x | Majorana)
                     + (pairing - hopping) * mv.x * (mv.y | Majorana))   # [] Majorana+ <- Majorana+
    forward_couplings = SparseExtensor(bond_coupling.broadcast_to((n_sites - 1,)), sites[:-1], sites[1:],
                                       (n_sites, n_sites))      # (Majorana+ <- Majorana+)[n_sites <- n_sites]
    # Reversing a coupling reverses its sign so the full bond map is skew-adjoint.
    return forward_couplings - forward_couplings.adjoint()


def wire_generator(couplings: SparseExtensor, potential: Scalar) -> SparseExtensor:
    # Chemical potential turns each site's two components in xy.
    local = -potential * mv.xy.commutator(Majorana)             # [...] (Majorana+ <- Majorana+)[V]
    return couplings + spdiag(local)


couplings = chain_couplings(n_sites, hopping, pairing)          # (Majorana+ <- Majorana+)[n_sites <- n_sites]

## Forming the end modes

Lowering the chemical potential inside the gate below twice the hopping drives the gated segment into the topological phase. Composing the skew generator with its adjoint yields a positive-semidefinite map whose lowest eigenfields trace the spectrum and the spatial Majorana weight across the transition.

In [ ]:
n_potentials = 200
n_energies = 8
chemical_potentials = np.linspace(outside_potential, inside_potential, n_potentials)   # [n_potentials]

In [ ]:
def gate_potential(
    sites: np.ndarray, left_end: np.ndarray, right_end: np.ndarray,
    inside_potential: np.ndarray, outside_potential: float, edge_width: float,
) -> Scalar:
    window = (np.tanh((sites - left_end) / edge_width) - np.tanh((sites - right_end) / edge_width)) / 2
    return mv.scalar((outside_potential + (inside_potential - outside_potential) * window)[..., None]).field()


def excitation_modes(generator: SparseExtensor, n_modes: int) -> tuple[Scalar, Majorana]:
    # Squaring the norm of the turning map gives squared excitation energies against the site identity as metric.
    _, states = generator.adjoint()(generator).eigh(0 * generator + Majorana, n_modes)   # [..., n_modes] Majorana+[V]
    # Taking the image norm avoids a square root of roundoff-sized negative eigenvalues at zero modes.
    energies = generator[..., None](states).scalar_norm_squared().sites.sum().square_root()   # [..., n_modes] Scalar+
    return energies, states


formation_potential = gate_potential(
    sites, np.array(left_end), np.array(right_end), chemical_potentials[:, None], outside_potential, edge_width,
)                                                               # [n_potentials] Scalar+[n_sites]
formation_energies, formation_states = excitation_modes(wire_generator(couplings, formation_potential), n_energies)   # [n_potentials, n_energies] Scalar+, Majorana+[n_sites]
# Averaging the two quadratures removes the arbitrary eigenvector orientation within the pair.
formation_density = formation_states[:, :2].scalar_norm_squared().mean(axis=-1)   # [n_potentials] Scalar+[n_sites]
save_figure(render.formation_figure(chemical_potentials, formation_energies, formation_density, hopping), "kitaev_formation");

Once the interior chemical potential crosses the dotted bulk threshold at twice the hopping, one pair of modes detaches from the continuum, drops to zero energy, and concentrates at the two gate boundaries.

## Separating the left and right end modes

Because the two end modes are nearly degenerate, the eigensolver returns arbitrary orthogonal combinations of them. Projecting the site-position operator onto that two-dimensional subspace separates the pair into one mode localized at the left boundary and one at the right boundary along a smooth gate trajectory.

In [ ]:
left_destination = 48.0
n_steps = 720
steps_per_frame = 3
n_frames = n_steps // steps_per_frame + 1
progress = np.arange(n_frames) * steps_per_frame / n_steps      # [n_frames]
site_positions = mv.scalar(sites[:, None]).field()              # [] Scalar+[n_sites]

In [ ]:
def smooth_schedule(progress: np.ndarray) -> np.ndarray:
    # Zero velocity and acceleration at both endpoints suppress sudden switching transients.
    return progress**3 * (10 + progress * (-15 + 6 * progress))


def localized_modes(states: Majorana, site_positions: Scalar) -> Majorana:
    # The position along the wire between every pair of the two modes acts as a 2x2 map on the mode subspace.
    position_elements = states[..., :, None].scalar_product(site_positions * states[..., None, :]).sites.sum()   # [..., 2, 2] Scalar+
    position_map = (position_elements * Scalar).field(0, 1)     # [...] (Scalar+ <- Scalar+)[2 <- 2]
    # The eigenvectors of this map select the left and right end combinations.
    _, directions = position_map.eigh()                         # [..., 2] Scalar+[2]
    return (directions.batch() * states[..., None, :]).sum(axis=-1)


moving_ends = left_end + (left_destination - left_end) * smooth_schedule(progress)   # [n_frames]
potentials = gate_potential(
    sites, moving_ends[:, None], np.array(right_end), np.array(inside_potential), outside_potential, edge_width,
)                                                               # [n_frames] Scalar+[n_sites]
_, boundary_modes = excitation_modes(wire_generator(couplings, potentials), 2)   # [n_frames, 2] Majorana+[n_sites]
localized = localized_modes(boundary_modes, site_positions)     # [n_frames, 2] Majorana+[n_sites]

## Adiabatic and fast gate transport

Starting in the initial left end mode, the field evolves through the moving gate at two driving durations. Evaluating the skew generator halfway through each step in a Cayley update keeps the time evolution orthogonal and second-order accurate.

In [ ]:
durations = np.array([180.0, 12.0])                             # [2]
frame_duration_ms = 40
midpoint_ends = left_end + (left_destination - left_end) * smooth_schedule((np.arange(n_steps) + 0.5) / n_steps)   # [n_steps]
midpoint_potentials = gate_potential(
    sites, midpoint_ends[:, None], np.array(right_end), np.array(inside_potential), outside_potential, edge_width,
)                                                               # [n_steps] Scalar+[n_sites]

In [ ]:
def cayley_step(couplings: SparseExtensor, state: Majorana, potential: Scalar, time_step: np.ndarray) -> Majorana:
    # A Cayley step of a skew map is orthogonal and preserves the mode norm.
    half_step = wire_generator(couplings, potential) * (time_step[..., None] / 2)   # [...] (Majorana+ <- Majorana+)[V <- V]
    return (Majorana - half_step).solve(state + half_step(state))


def boundary_share(modes: Majorana, state: Majorana) -> Scalar:
    overlap = state[..., None].scalar_product(modes).sites.sum()   # [..., 2] Scalar+
    return overlap.squared().sum(axis=-1)


In [ ]:
def transport_states(
    couplings: SparseExtensor, initial: Majorana, midpoint_potentials: Scalar,
    boundary_modes: Majorana, durations: np.ndarray, steps_per_frame: int,
) -> Iterator[tuple[Majorana, Scalar]]:
    n_speeds = len(durations)
    time_step = durations / len(midpoint_potentials)            # [n_speeds]
    state = initial.broadcast_to((n_speeds,))                   # [n_speeds] Majorana+[V]
    yield state, boundary_share(boundary_modes[0][None], state)
    for frame, block in enumerate(midpoint_potentials.reshape(-1, steps_per_frame), start=1):
        for potential in block:
            state = cayley_step(couplings, state, potential, time_step)
        yield state, boundary_share(boundary_modes[frame][None], state)


recorded = show_states(
    transport_states(couplings, localized[0, 0], midpoint_potentials, boundary_modes, durations, steps_per_frame),
    render.transport_animation, progress, potentials, localized[:, 0], hopping,
    name="kitaev_transport", duration_ms=frame_duration_ms, n_frames=n_frames,
);

Teal curves show the slow drive in the top strip and orange curves show the fast drive in the middle strip, each plotted over the lavender topological gate region against the dashed instantaneous end mode. Slow driving tracks the moving boundary with negligible bulk weight in the bottom strip, whereas fast driving radiates wavepackets into the interior of the segment.

In [ ]:
history = stack([state for state, _ in recorded])               # [n_frames, 2] Majorana+[n_sites]
shares = stack([share for _, share in recorded])                # [n_frames, 2] Scalar+
save_figure(render.transport_figure(progress, history, potentials, hopping), "kitaev_transport");

On the space-time panels, time runs upward and the dashed white contour marks the topological threshold; the slow drive on the left stays bound to the moving contour while the fast drive on the right bounces across the topological well.

## Hybridization as the ends approach

Bringing the two boundaries of the topological segment close together lets the exponential tails of the left and right Majorana modes overlap, lifting their shared fermion away from zero energy.

In [ ]:
n_separations = 200
separations = np.linspace(32.0, 4.0, n_separations)             # [n_separations]

In [ ]:
closing_potentials = gate_potential(
    sites, right_end - separations[:, None], np.array(right_end), np.array(inside_potential), outside_potential, edge_width,
)                                                               # [n_separations] Scalar+[n_sites]
closing_energies, closing_modes = excitation_modes(wire_generator(couplings, closing_potentials), 2)   # [n_separations, 2] Scalar+, Majorana+[n_sites]
splitting = closing_energies.mean(axis=-1)                      # [n_separations] Scalar+
closing_localized = localized_modes(closing_modes, site_positions)   # [n_separations, 2] Majorana+[n_sites]
show_animation(
    render.overlap_animation(separations, splitting, closing_potentials, closing_localized, hopping),
    "kitaev_overlap", frame_duration_ms, n_frames=n_separations, forward_then_back=True,
)

On the logarithmic energy axis, the lowest excitation grows exponentially with decreasing gate separation, oscillating as the overlapping Majorana tails interfere across the lattice.

In [ ]:
# checks
# Topological ground pair sits near zero energy while outside the segment the gap stays open.
assert formation_energies[-1, 0] < 1e-4 and formation_energies[0, 0] > 1.0
# Localized initial modes sit on opposite ends of the topological segment.
initial_centres = (localized[0].scalar_norm_squared() * site_positions).sites.sum()   # [2] Scalar+
assert initial_centres[1] - initial_centres[0] > 50
# Midpoint Cayley stepping preserves the Majorana state norm across every frame and speed.
norms = history.scalar_norm_squared().sites.sum()               # [n_frames, 2] Scalar+
assert ((norms - 1).abs() < 1e-10).all()
# Slow transport retains almost all weight in the boundary subspace, while fast transport leaks into the bulk.
assert shares[-1, 0] > 0.99 and shares[-1, 1] < 0.5
# Closing the separation lifts the hybridized splitting by orders of magnitude.
assert splitting[-1] > 100 * splitting[0]